<a href="https://colab.research.google.com/github/WellingtonRoque/MineracaoDados/blob/main/aulas/Aula07_Banco_de_Dados_SQL_Pandas_REVISADA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗄️ Aula 07 — Banco de Dados, SQL e Pandas

## 🔗 Integrando bancos de dados ao processo de Mineração de Dados

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** SQLite, Pandas

> **Importante:** este notebook foi estruturado para poder ser executado novamente no Google Colab sem gerar erros de tabela já existente ou chave primária duplicada.

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o papel dos bancos de dados na Mineração de Dados;
- Entender tabelas, registros, campos e chaves;
- Executar comandos SQL básicos;
- Utilizar `SELECT`, `WHERE`, `ORDER BY` e `GROUP BY`;
- Utilizar funções de agregação;
- Relacionar tabelas com `JOIN`;
- Integrar SQL com Pandas;
- Transformar consultas SQL em DataFrames;
- Preparar dados vindos de banco para análise;
- Aplicar o conhecimento ao projeto de Mineração de Dados.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. No projeto de avaliação, cada aluno/grupo trabalhará com seu próprio tema.

# 🏭 1. Onde estão os dados de uma empresa?

Nas aulas anteriores trabalhamos com:

```text
CSV
HTML
Web Scraping
DataFrames
ETL
```

No ambiente profissional, grande parte dos dados está armazenada em **bancos de dados**.

Imagine uma indústria:

```text
Sensores ─────────────┐
Sistema de produção ──┼──> BANCO DE DADOS
Manutenção ───────────┘
                           ↓
                    Mineração de Dados
```

Para minerar esses dados, precisamos saber **consultá-los**.

# 🧠 2. Conceitos fundamentais

Um banco de dados relacional é organizado em **tabelas**.

### Tabela `motores`

| id | motor | modelo | potencia |
|---:|---|---|---:|
| 1 | M001 | MX-100 | 7.5 |
| 2 | M002 | MX-100 | 6.5 |
| 3 | M003 | MX-200 | 9.0 |

### Tabela `leituras`

| id | motor_id | temperatura | vibracao | corrente |
|---:|---:|---:|---:|---:|
| 1 | 1 | 62.3 | 1.8 | 12.4 |
| 2 | 1 | 64.1 | 2.1 | 12.8 |
| 3 | 2 | 55.1 | 1.2 | 10.8 |

Uma tabela possui:

- **colunas** → atributos;
- **linhas** → registros;
- **chaves** → identificadores utilizados para relacionar dados.

# 💻 3. Criando um banco SQLite

Para não depender de instalação ou servidor externo, utilizaremos o **SQLite**.

Ele já está disponível no Python.

Em projetos profissionais, os mesmos conceitos podem ser aplicados a MySQL, PostgreSQL, SQL Server e outros bancos relacionais.

In [ ]:
import sqlite3
import pandas as pd

conexao = sqlite3.connect("industria4.db")

# Ativa a verificação das chaves estrangeiras no SQLite
conexao.execute("PRAGMA foreign_keys = ON")

print("Banco de dados conectado!")

# 🔁 4. Preparando o notebook para reexecução

No Google Colab é comum executar a mesma célula mais de uma vez.

Por isso, utilizaremos dois padrões:

- `CREATE TABLE IF NOT EXISTS` → não tenta recriar uma tabela que já existe;
- `INSERT OR IGNORE` → não reinsere um registro que já possui a mesma chave primária.

Além disso, cada etapa que altera o banco termina com:

```python
conexao.commit()
```

Assim, o notebook pode ser executado novamente sem produzir os erros:

```text
table ... already exists
UNIQUE constraint failed
```

### 🧹 Opcional — começar o banco completamente do zero

**Não execute esta célula durante a aula, a menos que queira apagar os dados existentes.**

As tabelas filhas devem ser removidas antes das tabelas que possuem a chave referenciada.

In [ ]:
# OPCIONAL: apaga todo o conteúdo e recria o banco do zero.
# Execute somente se quiser reiniciar completamente o exercício.

conexao.execute("DROP TABLE IF EXISTS manutencoes")
conexao.execute("DROP TABLE IF EXISTS leituras")
conexao.execute("DROP TABLE IF EXISTS motores")
conexao.commit()

print("Tabelas removidas. O banco está pronto para ser recriado.")

# 🏗️ 5. Criando as tabelas

Vamos criar três tabelas:

```text
motores
leituras
manutencoes
```

Elas representam diferentes fontes de informação da nossa indústria.

In [ ]:
conexao.execute("""
CREATE TABLE IF NOT EXISTS motores (
    id INTEGER PRIMARY KEY,
    motor TEXT NOT NULL,
    modelo TEXT,
    fabricante TEXT,
    potencia_nominal REAL,
    linha TEXT
)
""")

conexao.commit()

print("Tabela motores pronta.")

In [ ]:
conexao.execute("""
CREATE TABLE IF NOT EXISTS leituras (
    id INTEGER PRIMARY KEY,
    motor_id INTEGER,
    data_hora TEXT,
    corrente REAL,
    tensao REAL,
    vibracao REAL,
    temperatura REAL,
    rpm INTEGER,
    FOREIGN KEY (motor_id) REFERENCES motores(id)
)
""")

conexao.commit()

print("Tabela leituras pronta.")

In [ ]:
conexao.execute("""
CREATE TABLE IF NOT EXISTS manutencoes (
    id INTEGER PRIMARY KEY,
    motor_id INTEGER,
    data_manutencao TEXT,
    horas_operacao INTEGER,
    status TEXT,
    FOREIGN KEY (motor_id) REFERENCES motores(id)
)
""")

conexao.commit()

print("Tabela manutencoes pronta.")

### Verificando as tabelas

Vamos conferir se as três tabelas existem no banco.

In [ ]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conexao)

# 📥 6. Inserindo os dados

Os dados representam quatro motores da nossa indústria.

O `INSERT OR IGNORE` foi utilizado para que a célula possa ser executada novamente sem gerar erro de chave primária duplicada.

In [ ]:
motores = [
    (1, "M001", "MX-100", "MotorTech", 7.5, "Linha A"),
    (2, "M002", "MX-100", "MotorTech", 6.5, "Linha A"),
    (3, "M003", "MX-200", "PowerMotor", 9.0, "Linha B"),
    (4, "M004", "MX-300", "PowerMotor", 11.0, "Linha B")
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO motores
    (id, motor, modelo, fabricante, potencia_nominal, linha)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    motores
)

conexao.commit()

print("Dados de motores inseridos/verificados.")

In [ ]:
leituras = [
    (1, 1, "2026-03-01 08:00", 12.4, 380, 1.8, 62.3, 1750),
    (2, 1, "2026-03-01 09:00", 12.8, 379, 2.1, 64.1, 1748),
    (3, 1, "2026-03-01 10:00", 13.1, 380, 2.5, 66.2, 1745),
    (4, 2, "2026-03-01 08:00", 10.8, 380, 1.2, 55.1, 1752),
    (5, 2, "2026-03-01 09:00", 11.0, 381, 1.3, 56.0, 1750),
    (6, 2, "2026-03-01 10:00", 11.2, 380, 1.5, 57.4, 1748),
    (7, 3, "2026-03-01 08:00", 14.1, 382, 2.4, 67.2, 1740),
    (8, 3, "2026-03-01 09:00", 14.3, 381, 2.6, 68.4, 1738),
    (9, 3, "2026-03-01 10:00", 14.8, 380, 2.9, 70.1, 1735),
    (10, 4, "2026-03-01 08:00", 16.2, 381, 3.1, 72.4, 1728),
    (11, 4, "2026-03-01 09:00", 16.8, 380, 3.5, 75.2, 1724),
    (12, 4, "2026-03-01 10:00", 17.4, 379, 3.8, 78.1, 1720)
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO leituras
    (id, motor_id, data_hora, corrente, tensao, vibracao, temperatura, rpm)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """,
    leituras
)

conexao.commit()

print("Dados de leituras inseridos/verificados.")

In [ ]:
manutencoes = [
    (1, 1, "2026-02-15", 1254, "Em dia"),
    (2, 2, "2026-02-20", 824, "Em dia"),
    (3, 3, "2026-01-18", 2104, "Atenção"),
    (4, 4, "2026-01-10", 2980, "Atenção")
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO manutencoes
    (id, motor_id, data_manutencao, horas_operacao, status)
    VALUES (?, ?, ?, ?, ?)
    """,
    manutencoes
)

conexao.commit()

print("Dados de manutenção inseridos/verificados.")

### Conferindo a quantidade de registros

Esta etapa ajuda a verificar se os dados foram carregados corretamente.

In [ ]:
resumo = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM motores) AS total_motores,
    (SELECT COUNT(*) FROM leituras) AS total_leituras,
    (SELECT COUNT(*) FROM manutencoes) AS total_manutencoes
""", conexao)

resumo

# 🔎 7. SELECT — Consultando dados

O comando mais básico do SQL é `SELECT`.

In [ ]:
consulta = """
SELECT *
FROM motores
"""

pd.read_sql_query(consulta, conexao)

In [ ]:
consulta = """
SELECT motor, modelo, potencia_nominal
FROM motores
"""

pd.read_sql_query(consulta, conexao)

# 🎯 8. WHERE — Filtrando dados

O `WHERE` permite selecionar registros que atendam a uma condição.

In [ ]:
consulta = """
SELECT *
FROM motores
WHERE linha = 'Linha A'
"""

pd.read_sql_query(consulta, conexao)

In [ ]:
consulta = """
SELECT motor, potencia_nominal
FROM motores
WHERE potencia_nominal > 7
"""

pd.read_sql_query(consulta, conexao)

Podemos utilizar operadores como:

```text
=    >    <    >=    <=    <>    AND    OR
```

# ↕️ 9. ORDER BY — Ordenando resultados

In [ ]:
consulta = """
SELECT motor, potencia_nominal
FROM motores
ORDER BY potencia_nominal DESC
"""

pd.read_sql_query(consulta, conexao)

`DESC` → decrescente  
`ASC` → crescente

# 📊 10. Funções de agregação

SQL possui funções para realizar cálculos:

```text
COUNT()
AVG()
SUM()
MIN()
MAX()
```

In [ ]:
consulta = """
SELECT AVG(temperatura) AS temperatura_media
FROM leituras
"""

pd.read_sql_query(consulta, conexao)

In [ ]:
consulta = """
SELECT
    MIN(temperatura) AS temperatura_minima,
    MAX(temperatura) AS temperatura_maxima,
    AVG(temperatura) AS temperatura_media,
    AVG(vibracao) AS vibracao_media
FROM leituras
"""

pd.read_sql_query(consulta, conexao)

# 👥 11. GROUP BY

Agora queremos saber a temperatura média de **cada motor**.

In [ ]:
consulta = """
SELECT
    motor_id,
    AVG(temperatura) AS temperatura_media
FROM leituras
GROUP BY motor_id
ORDER BY temperatura_media DESC
"""

pd.read_sql_query(consulta, conexao)

O `GROUP BY` é muito importante em Mineração de Dados porque permite resumir conjuntos de registros.

Exemplos:

- média por produto;
- vendas por região;
- consumo por cliente;
- temperatura média por equipamento;
- quantidade de ocorrências por categoria.

# 🔗 12. JOIN — Relacionando tabelas

Até agora `leituras` possui apenas `motor_id`.

Queremos descobrir o nome do motor. Para isso, precisamos relacionar as tabelas.

In [ ]:
consulta = """
SELECT
    m.motor,
    m.modelo,
    l.data_hora,
    l.temperatura,
    l.vibracao,
    l.corrente
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
"""

pd.read_sql_query(consulta, conexao)

O `JOIN` permite transformar:

```text
Tabela A
   +
Tabela B
   ↓
Informação integrada
```

Isso se conecta diretamente com o que estudamos sobre **ETL**.

# 🧠 13. Criando uma base para Mineração de Dados

Agora vamos produzir uma tabela integrada com informações das três tabelas.

In [ ]:
consulta = """
SELECT
    m.motor,
    m.modelo,
    m.fabricante,
    m.linha,
    l.data_hora,
    l.corrente,
    l.tensao,
    l.vibracao,
    l.temperatura,
    l.rpm,
    man.horas_operacao,
    man.status AS status_manutencao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
LEFT JOIN manutencoes AS man
    ON m.id = man.motor_id
ORDER BY m.motor, l.data_hora
"""

dados_mineracao = pd.read_sql_query(consulta, conexao)

dados_mineracao

Agora temos uma base que pode ser utilizada nas próximas etapas de análise.

```text
Banco de Dados
      ↓
SQL
      ↓
Consulta
      ↓
Pandas
      ↓
DataFrame
      ↓
Análise / Mineração
```

# 🐼 14. SQL + Pandas

SQL é excelente para:

- selecionar;
- filtrar;
- relacionar;
- agrupar.

Pandas é excelente para:

- limpar;
- transformar;
- explorar;
- analisar;
- visualizar.

As duas ferramentas podem trabalhar juntas.

In [ ]:
dados_mineracao.info()

In [ ]:
dados_mineracao.describe()

# 🔎 15. Investigando o problema

Vamos procurar motores que apresentem temperatura acima de 70 °C.

In [ ]:
consulta = """
SELECT
    m.motor,
    l.data_hora,
    l.temperatura,
    l.vibracao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
WHERE l.temperatura > 70
ORDER BY l.temperatura DESC
"""

pd.read_sql_query(consulta, conexao)

Agora podemos combinar duas condições:

```text
temperatura > 70
E
vibracao > 3
```

In [ ]:
consulta = """
SELECT
    m.motor,
    l.data_hora,
    l.temperatura,
    l.vibracao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
WHERE l.temperatura > 70
  AND l.vibracao > 3
ORDER BY l.temperatura DESC
"""

pd.read_sql_query(consulta, conexao)

# 📝 16. Exercícios

## Exercício 1 — Consulta básica

Liste todos os motores mostrando:

- motor;
- modelo;
- fabricante;
- potência.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 2 — Filtro

Liste somente os motores da `Linha B`.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 3 — Condição numérica

Liste os motores com potência nominal maior que 8 kW.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 4 — Ordenação

Liste todos os motores ordenados pela potência, da maior para a menor.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 5 — Agregação

Calcule:

- temperatura mínima;
- temperatura máxima;
- temperatura média;
- vibração média.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 6 — GROUP BY

Calcule a temperatura média de cada motor e ordene do maior para o menor.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 7 — JOIN

Crie uma consulta que mostre:

- motor;
- modelo;
- temperatura;
- vibração;
- corrente.

Use `JOIN`.

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 8 — Problema

Encontre os registros em que:

```text
temperatura > 70
OU
vibracao > 3
```

In [ ]:
# Sua resposta

# Escreva sua consulta SQL aqui.

## Exercício 9 — SQL + Pandas

Execute uma consulta SQL e armazene o resultado em um DataFrame chamado `df_alertas`.

Depois utilize Pandas para mostrar informações estatísticas desse DataFrame.

In [ ]:
# Sua resposta

# Exemplo de estrutura:
# consulta = """ ... """
# df_alertas = pd.read_sql_query(consulta, conexao)
# df_alertas.describe()

## Exercício 10 — Investigação

Qual motor apresenta o maior valor de temperatura?

Qual apresenta a maior vibração?

Qual deles merece maior atenção?

Justifique utilizando os dados.

In [ ]:
# Sua resposta

# Escreva sua análise aqui.

# 🔎 17. Desafio — Criando sua consulta de mineração

Crie uma consulta SQL que produza uma tabela contendo somente os registros que poderiam representar uma situação de atenção.

Você deve definir seus próprios critérios utilizando pelo menos **duas variáveis**.

Depois:

1. execute a consulta;
2. transforme o resultado em DataFrame;
3. faça uma análise;
4. explique por que escolheu esses critérios.

In [ ]:
# Desenvolva sua solução aqui.

# 🚀 18. Aplicação no seu projeto

Agora pense no projeto de avaliação.

Responda:

| Pergunta | Resposta |
|---|---|
| Meu projeto utilizará banco de dados? | ... |
| Qual banco? | ... |
| Quais tabelas serão necessárias? | ... |
| Qual será a chave principal? | ... |
| Quais tabelas precisarão ser relacionadas? | ... |
| Quais consultas serão importantes para a análise? | ... |
| Quais indicadores quero calcular? | ... |

### Desafio

Desenhe, mesmo que de forma simples, as principais tabelas do seu projeto.

Exemplo:

```text
CLIENTE
   │
   └──── VENDA
             │
             └──── PRODUTO
```

O objetivo é começar a pensar na estrutura dos dados antes de aplicar técnicas de Mineração de Dados.

In [ ]:
# Planejamento do banco de dados do seu projeto

# Escreva ou desenhe sua proposta aqui.

# 📌 19. Checklist da Aula

- [ ] Entendo o papel do banco de dados na Mineração de Dados;
- [ ] Sei criar tabelas SQLite;
- [ ] Sei inserir dados;
- [ ] Sei utilizar `SELECT`;
- [ ] Sei utilizar `WHERE`;
- [ ] Sei utilizar `ORDER BY`;
- [ ] Sei utilizar funções de agregação;
- [ ] Sei utilizar `GROUP BY`;
- [ ] Entendo o funcionamento do `JOIN`;
- [ ] Sei executar SQL e obter um DataFrame;
- [ ] Sei combinar SQL e Pandas;
- [ ] Consigo pensar nas tabelas necessárias para meu projeto.

# 🎯 Conclusão

Nesta aula avançamos:

```text
Aula 3 → Pandas
Aula 4 → Limpeza
Aula 5 → ETL
Aula 6 → Web Scraping
Aula 7 → Banco de Dados + SQL
```

Agora já sabemos trabalhar com dados provenientes de:

```text
Arquivos
Web
Banco de Dados
```

Na próxima aula vamos utilizar esses dados para responder:

> **O que os dados estão nos dizendo?**

Entraremos em **Análise Exploratória de Dados (EDA)**.

## 🔒 Encerrando a conexão

Ao finalizar o notebook, podemos fechar a conexão com o banco.

In [ ]:
conexao.close()

print("Conexão encerrada.")